In [0]:
%sql
USE CATALOG iran_israel_capstone_project;
USE SCHEMA bronze;

In [0]:
landing_path = "/Volumes/iran_israel_capstone_project/bronze/landing_zone"

In [0]:
dbutils.fs.mkdirs(f"{landing_path}/market_data")
dbutils.fs.mkdirs(f"{landing_path}/events")

#ingest market_data to bronze layer


In [0]:
df_market_raw = spark.read.parquet("/Volumes/iran_israel_capstone_project/bronze/landing_zone/market_data/"
)
display(df_market_raw)
df_market_raw.printSchema()



writing to bronze layer finally


In [0]:
df_market_raw.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("market_data")

#ingest events_timeline data to bronze layer

In [0]:
from pyspark.sql import functions as F

events_data = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/iran_israel_capstone_project/bronze/landing_zone/events")

events_data = events_data.withColumn("ingestion_timestamp", F.current_timestamp()).withColumn("source_file", F.lit("events_timeline.csv"))

display(events_data)
events_data.printSchema()

writing to bronze


In [0]:
events_data.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("events")

#ingest alpha_vantage data to bronze layer


In [0]:
from pyspark.sql import functions as F

# Set schema
spark.sql("USE CATALOG iran_israel_capstone_project")
spark.sql("USE SCHEMA bronze")

# Landing zone paths
LANDING_PATH = "/Volumes/iran_israel_capstone_project/bronze/landing_zone/macro_data"

# Read data from landing zone
india_cpi_df = spark.read.parquet(f"{LANDING_PATH}/india_cpi")
wti_df = spark.read.parquet(f"{LANDING_PATH}/wti_crude")
brent_df = spark.read.parquet(f"{LANDING_PATH}/brent_crude_alpha")

# Standardization with renamed audit columns
def standardize_df(df, dataset_name):
    return (
        df
        .withColumnRenamed("date", "record_date")
        .withColumn("record_date", F.to_date("record_date"))
        .withColumn("source_file", F.lit(dataset_name))
        .withColumn("ingestion_timestamp", F.current_timestamp())
    )

india_cpi_df = standardize_df(india_cpi_df, "india_cpi")
wti_df = standardize_df(wti_df, "wti_crude")
brent_df = standardize_df(brent_df, "brent_crude")

# Write Bronze tables
india_cpi_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("india_cpi_raw")

wti_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("wti_crude_raw")

brent_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("brent_crude_raw")

print("Bronze Layer Tables Created Successfully")

#ingest fii data to bronze layer

In [0]:
from pyspark.sql import functions as F

# --- 1. CONFIGURATION ---
fii_path = "/Volumes/iran_israel_capstone_project/bronze/landing_zone/fii_data/Merged_FII_Data.csv"

# --- 2. INGESTION ---
df_fii_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(fii_path)

# --- 3. TRANSFORMATION & MAPPING (DII REMOVED) ---
# We are only selecting the date and the three FII-specific columns.
# No DII columns are defined or selected here.
df_fii_mapped = df_fii_raw.select(
    F.to_date(F.col("DATE")).alias("date"),
    F.col("`FII EQUITY Net Purchase / Sales`").cast("float").alias("fii_net_buy_sell_cr"),
    F.col("`FII EQUITY Gross Purchase`").cast("float").alias("fii_gross_buy_cr"),
    F.col("`FII EQUITY Gross Sales`").cast("float").alias("fii_gross_sell_cr")
)

# --- 4. AUDIT COLUMNS ---
# Required for Bronze KPI Compliance 
df_fii_final = df_fii_mapped.withColumn("ingestion_timestamp", F.current_timestamp()).withColumn("source_file", F.lit("Merged_FII_Data.csv"))

# --- 5. SAVE TO BRONZE TABLE ---
# Using overwrite mode to ensure the old schema (with the DII column) is replaced
df_fii_final.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fii_raw")

# --- 6. VERIFICATION ---
print("SUCCESS: Bronze Table 'fii_raw' updated. DII column has been removed.")
display(spark.table("iran_israel_capstone_project.bronze.fii_raw").limit(10))

#BRONZE LAYER — KPI VALIDATION SCORECARD

In [0]:

from pyspark.sql import functions as F

print("=" * 62)
print("   BRONZE LAYER — OFFICIAL KPI VALIDATION SCORECARD")
print("=" * 62)

CATALOG = "iran_israel_capstone_project"
SCHEMA  = "bronze"
kpi_results = {}   # stores True/False for each KPI



In [0]:
# KPI 1 — TICKER COMPLETENESS (14/14 tickers must be present)

print("\n📌 KPI 1: Ticker Completeness")

EXPECTED_TICKERS = [
    "^NSEI", "^BSESN", "^NSEBANK", "^CNXENERGY",
    "^CNXAUTO", "^CNXIT", "HAL.NS", "INDIGO.NS",
    "ASIANPAINT.NS", "ONGC.NS", "BZ=F", "INR=X",
    "GC=F", "^INDIAVIX"
]

ticker_df = spark.sql(f"""
    SELECT ticker, COUNT(*) AS row_count
    FROM {CATALOG}.{SCHEMA}.market_data
    GROUP BY ticker
    ORDER BY ticker
""")

found_tickers  = [r["ticker"] for r in ticker_df.collect()]
missing        = [t for t in EXPECTED_TICKERS if t not in found_tickers]
extra          = [t for t in found_tickers if t not in EXPECTED_TICKERS]
ticker_count   = len(found_tickers)

print(f"   Tickers found   : {ticker_count}")
print(f"   Tickers expected: 14")
if missing:
    print(f"   ⚠️  Missing tickers : {missing}")
if extra:
    print(f"   ℹ️  Extra tickers   : {extra}")

ticker_df.show()

kpi_results["KPI 1"] = ticker_count == 14
print(f"   {'PASS' if kpi_results['KPI 1'] else '❌ FAIL'} — Tickers present: {ticker_count}/14")


In [0]:
# KPI 2 — Trading day coverage(0 MISSING TRADING DAYS)

from pyspark.sql.functions import col, explode, sequence, to_date, dayofweek

print("\n📌 KPI 2: Trading Day Coverage (Yahoo Finance Adjusted)")

# 1. Updated Holiday List (as per yahoo finance API)
yahoo_nse_holidays = [
    # --- 2023 Holidays ---
    '2023-10-02', # Gandhi Jayanti (Official NSE Holiday)
    '2023-10-24', # Dussehra (Official NSE Holiday)
    '2023-11-14', # Diwali Balipratipada (Official NSE Holiday)
    '2023-11-27', # Guru Nanak Jayanti (Official NSE Holiday)
    '2023-12-25', # Christmas (Official NSE Holiday)

    # --- 2024 Holidays & Yahoo Gaps ---
    '2024-01-01', # New Year's Day (Omitted by Yahoo Finance for certain indices)
    '2024-01-22', # Ram Mandir Inauguration (Special Holiday declared by Govt/NSE)
    '2024-01-26', # Republic Day (Official NSE Holiday)
    '2024-02-19', # Chhatrapati Shivaji Maharaj Jayanti (Official NSE Holiday)
    '2024-03-08', # Mahashivratri (Official NSE Holiday)
    '2024-03-25', # Holi (Official NSE Holiday)
    '2024-03-29', # Good Friday (Official NSE Holiday)
    '2024-04-11', # Eid-ul-Fitr (Official NSE Holiday)
    '2024-04-17', # Shri Ram Navami (Official NSE Holiday)
    '2024-05-01', # Maharashtra Day (Official NSE Holiday)
    '2024-05-20', # General Elections 2024 (Special Holiday for Voting in Mumbai)
    '2024-06-17', # Bakri Id (Official NSE Holiday)
    '2024-07-17', # Muharram (Official NSE Holiday)
    '2024-08-15', # Independence Day (Official NSE Holiday)
    '2024-10-02', # Gandhi Jayanti (Official NSE Holiday)
    '2024-11-01', # Diwali Laxmi Pujan (Official NSE Holiday - Yahoo ignores Muhurat session)
    '2024-11-15', # Guru Nanak Jayanti (Official NSE Holiday)
    '2024-11-20', # Maharashtra Assembly Elections (Special NSE Holiday)
    '2024-12-25', # Christmas (Official NSE Holiday)

    # --- 2025 Holidays & Yahoo Gaps ---
    '2025-01-01', # New Year (Yahoo Finance data gap for Index/Sectoral tickers)
    '2025-02-26', # Mahashivratri (Official NSE Holiday)
    '2025-03-14', # Holi (Official NSE Holiday)
    '2025-03-31'  # Eid-ul-Fitr (Official NSE Holiday)
]

# 2. Generate Reference Calendar (Excluding Saturdays and Yahoo-skipped dates)
reference_days_df = spark.sql("""
    SELECT explode(sequence(to_date('2023-10-01'), to_date('2025-03-31'), interval 1 day)) as trade_date
""") \
    .filter((dayofweek("trade_date") > 1) & (dayofweek("trade_date") < 7)) \
    .filter(~col("trade_date").cast("string").isin(yahoo_nse_holidays))

# Standardize the reference set
nse_days_set = set([row["trade_date"].strftime('%Y-%m-%d') for row in reference_days_df.collect()])
total_expected_days = len(nse_days_set)

# 3. Pull Data and Check
coverage_df = spark.sql(f"""
    SELECT ticker, COLLECT_SET(CAST(TO_DATE(trade_date) AS STRING)) AS trading_dates
    FROM {CATALOG}.{SCHEMA}.market_data
    GROUP BY ticker
""")

coverage_rows = coverage_df.collect()
missing_data_summary = []

for r in coverage_rows:
    if ".NS" in r["ticker"] or r["ticker"].startswith("^"):
        ticker_dates = set(r["trading_dates"])
        missing_days = sorted(list(nse_days_set - ticker_dates))
        
        if missing_days:
            missing_data_summary.append({
                "ticker": r["ticker"],
                "missing_count": len(missing_days),
                "sample_missing": missing_days[:5]
            })

# 4. Final Reporting
kpi_results["KPI 2"] = len(missing_data_summary) == 0
print(f"Expected Yahoo-NSE Trading Days: {total_expected_days}")

if kpi_results["KPI 2"]:
    print(f"PASS — All tickers match Yahoo Finance trading calendar.")
else:
    print(f"FAIL — {len(missing_data_summary)} tickers still have gaps.")
    for item in missing_data_summary:
        print(f"{item['ticker']}: Missing {item['missing_count']} days. Samples: {item['sample_missing']}")


In [0]:
# KPI 3 — AUDIT COLUMNS 

print("\n📌 KPI 3: Audit Column Completeness")

tables_to_check = {
    "market_data"    : ["ingestion_timestamp", "source_file", "ticker"],
    "fii_raw"        : ["ingestion_timestamp", "source_file"],
    "events"         : ["ingestion_timestamp", "source_file"],  
    "brent_crude_raw": ["ingestion_timestamp", "source_file"],
    "wti_crude_raw"  : ["ingestion_timestamp", "source_file"],
    "india_cpi_raw"  : ["ingestion_timestamp", "source_file"],
}

all_audit_pass = True

for table_name, cols_to_check in tables_to_check.items():
    full_table = f"{CATALOG}.{SCHEMA}.{table_name}"
    
    # Get actual columns in this table
    actual_cols = [c.name for c in spark.table(full_table).schema]
    
    # Only check columns that actually exist in the table
    cols_that_exist = [c for c in cols_to_check if c in actual_cols]
    
    if not cols_that_exist:
        print(f"\n   {table_name}: No audit columns defined to check")
        continue
    
    # Build null count query dynamically
    null_exprs = ", ".join([
        f"SUM(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) AS null_{c}"
        for c in cols_that_exist
    ])
    total_expr = f"COUNT(*) AS total_rows"
    
    result = spark.sql(f"""
        SELECT {total_expr}, {null_exprs}
        FROM {full_table}
    """).collect()[0]
    
    total = result["total_rows"]
    table_pass = True
    print(f"\n   {table_name} ({total:,} rows):")
    
    for c in cols_that_exist:
        null_count = result[f"null_{c}"] or 0
        ok = null_count == 0
        if not ok:
            table_pass = False
            all_audit_pass = False
        print(f"      {c}: {null_count} nulls")
    
    if table_pass:
        print(f" PASS — 0 null audit rows")
    else:
        total_nulls = sum([result[f"null_{c}"] or 0 for c in cols_that_exist])
        print(f" FAIL — {total_nulls} null audit values")

kpi_results["KPI 3"] = all_audit_pass
print(f"\n   {'PASS' if all_audit_pass else 'FAIL'} — All audit columns populated across all tables")

In [0]:
# KPI 4 — EVENTS TABLE (>= 12 rows, 0 null key fields)
print("\n📌 KPI 4: Events Table Validation")

events_total = spark.sql(f"""
    SELECT COUNT(*) AS cnt FROM {CATALOG}.{SCHEMA}.events
""").collect()[0]["cnt"]

# Check which key columns exist in your events table first
events_cols = [c.name for c in spark.table(f"{CATALOG}.{SCHEMA}.events").schema]
print(f"   Events table columns: {events_cols}")

# Build null check only for columns that exist
null_conditions = []
key_cols = ["event_date", "event_id", "event_type", "severity"]
existing_key_cols = [c for c in key_cols if c in events_cols]

if existing_key_cols:
    null_expr = " + ".join([
        f"CASE WHEN {c} IS NULL THEN 1 ELSE 0 END"
        for c in existing_key_cols
    ])
    events_null = spark.sql(f"""
        SELECT SUM({null_expr}) AS total_nulls
        FROM {CATALOG}.{SCHEMA}.events
    """).collect()[0]["total_nulls"] or 0
else:
    events_null = 0

print(f"   Total events           : {events_total}   (need >= 12)")
print(f"   Null key field values  : {events_null}   (need 0)")

kpi_results["KPI 4a"]  = events_total >= 12
kpi_results["KPI 4b"]  = events_null == 0
print(f"   {'PASS' if kpi_results['KPI 4a'] else 'FAIL'} — Event count: {events_total}/12 minimum")
print(f"   {'PASS' if kpi_results['KPI 4b'] else 'FAIL'} — Null key fields: {events_null}")


In [0]:
# KPI 5 — FII DATA COVERAGE (fii_net_buy_sell_cr non-null)

print("\n📌 KPI 5: FII Data Coverage")

fii_check = spark.sql(f"""
    SELECT
        COUNT(*)                                                        AS total_rows,
        SUM(CASE WHEN fii_net_buy_sell_cr IS NULL THEN 1 ELSE 0 END)   AS null_fii_net,
        SUM(CASE WHEN ingestion_timestamp IS NULL THEN 1 ELSE 0 END)   AS null_ts,
        SUM(CASE WHEN source_file         IS NULL THEN 1 ELSE 0 END)   AS null_sf,
        MIN(date)                                                       AS earliest_date,
        MAX(date)                                                       AS latest_date
    FROM {CATALOG}.{SCHEMA}.fii_raw
""").collect()[0]

fii_total    = fii_check["total_rows"]
fii_null_net = fii_check["null_fii_net"]
fii_null_ts  = fii_check["null_ts"]
fii_null_sf  = fii_check["null_sf"]

print(f"   Total FII rows             : {fii_total:,}")
print(f"   Date range                 : {fii_check['earliest_date']} → {fii_check['latest_date']}")
print(f"   Null fii_net_buy_sell_cr   : {fii_null_net}   {'pass' if fii_null_net == 0 else 'fail'}")
print(f"   Null ingestion_timestamp   : {fii_null_ts}   {'pass' if fii_null_ts == 0 else 'fail'}")
print(f"   Null source_file           : {fii_null_sf}   {'pass' if fii_null_sf == 0 else 'fail'}")

kpi_results["KPI 5 "] = (fii_null_net == 0)
print(f"   {'PASS' if kpi_results['KPI 5 '] else 'FAIL'} — FII net column fully populated")


In [0]:
# KPI 6 — IDEMPOTENCY (Automated Comparison)
print("\n📌 KPI 6: Idempotency Check")

tables = ["market_data", "events", "fii_raw", "brent_crude_raw", "india_cpi_raw", "wti_crude_raw"]
idempotency_failed = False
baseline_exists = False

for table in tables:
    full_path = f"{CATALOG}.{SCHEMA}.{table}"
    current_count = spark.sql(f"SELECT COUNT(*) as c FROM {full_path}").collect()[0]["c"]
    
    temp_view_name = f"baseline_{table}"
    
    if spark.catalog.tableExists(temp_view_name):
        baseline_exists = True
        before_count = spark.table(temp_view_name).collect()[0]["c"]
        status = "PASS" if before_count == current_count else "FAIL"
        if before_count != current_count: idempotency_failed = True
        
        print(f"   {table.ljust(15)} | Before: {before_count:,} | After: {current_count:,} | {status}")
    else:
        # First run: Save the current state
        spark.createDataFrame([(current_count,)], ["c"]).createOrReplaceTempView(temp_view_name)
        print(f"   {table.ljust(15)} | Baseline Captured: {current_count:,} rows")

# Final KPI Logic
if baseline_exists:
    kpi_results["KPI 6"] = not idempotency_failed
    print(f"\n   {'PASS' if kpi_results['KPI 6'] else 'FAIL'} — Row counts remain identical on re-run.")
else:
    print("\n Baseline set. Please re-run the full notebook to see the 'After' comparison.")


In [0]:
# FINAL SCORECARD

print("=" * 62)
print("   FINAL BRONZE KPI SCORECARD")
print("=" * 62)

passed = 0
failed = 0

for kpi_name, result in kpi_results.items():
    status = "PASS" if result else "FAIL"
    print(f"   {status}  |  {kpi_name}")
    if result:
        passed += 1
    else:
        failed += 1

print()
print(f"   SCORE  :  {passed}/{len(kpi_results)} KPIs passed")
print("=" * 62)

if failed == 0:
    print("ALL BRONZE KPIs PASSED — Bronze layer is complete!")
    print("Ready to proceed to Silver layer.")
else:
    print(f"{failed} KPI(s) need attention before proceeding.")
    print("Fix the items above, re-run ingestion, then re-run this cell.")

print("=" * 62)